# Recomendación basada en contenido: TF-IDF

TF-IDF es un primer modelo transparente y eficiente para títulos, descripciones, autores y géneros: representa texto como una matriz dispersa y permite similitud coseno mediante producto escalar sobre vectores normalizados. No utiliza ratings globales Goodreads como características.

Las evaluaciones usan únicamente perfiles construidos con entrenamiento y candidatos muestreados persistidos. No equivalen al ranking sobre el catálogo completo.

In [1]:
from pathlib import Path
import json, sys
import joblib, numpy as np, pandas as pd, scipy.sparse as sp, torch
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display, Markdown
ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'ml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from ml.content import build_content_frame, content_candidate_scores
from ml.recommendation import make_candidates, ranking_metrics, score_candidates
from ml.collaborative import MatrixFactorization
PROCESSED, ML = ROOT / 'data' / 'processed', ROOT / 'data' / 'ml'
ARTIFACTS = ROOT / 'artifacts' / 'content_tfidf'; ARTIFACTS.mkdir(parents=True, exist_ok=True)
SEED, RELEVANCE, TOP_K, MAX_USERS, NEGATIVES = 2026, 4, 10, 5000, 1000
sns.set_theme(style='whitegrid')


## 1. Calidad y representación bibliográfica
Se conservan libros sin descripción: título, autores, etiquetas temáticas, origen e idioma aportan señales alternativas. Se excluyen etiquetas sociales conocidas como `to-read` o `currently-reading` antes de construir texto.

In [2]:
books = pd.read_parquet(PROCESSED / 'books.parquet'); authors = pd.read_parquet(PROCESSED / 'authors.parquet')
book_authors = pd.read_parquet(PROCESSED / 'book_authors.parquet'); genres = pd.read_parquet(PROCESSED / 'book_genres.parquet'); source_genres = pd.read_parquet(PROCESSED / 'book_source_genres.parquet')
content = build_content_frame(books, book_authors, authors, genres, source_genres).sort_values('book_id').reset_index(drop=True)
quality = pd.DataFrame({'campo': ['descripción', 'autor', 'género temático', 'texto final'], 'cobertura_pct': [100 * content.description.ne('').mean(), 100 * content.authors_text.ne('').mean(), 100 * content.genres_text.ne('').mean(), 100 * content.content_text.ne('').mean()]})
display(quality.round(2))
display(Markdown('**Interpretación.** La representación no descarta faltantes: los libros sin descripción mantienen título y metadatos relacionales. TF-IDF no entiende semántica profunda ni sinónimos, por lo que es un baseline explicable, no un modelo lingüístico final.'))


,campo,cobertura_pct
0,descripción,81.88
1,autor,100.00
2,género temático,100.00
3,texto final,100.00


**Interpretación.** La representación no descarta faltantes: los libros sin descripción mantienen título y metadatos relacionales. TF-IDF no entiende semántica profunda ni sinónimos, por lo que es un baseline explicable, no un modelo lingüístico final.

## 2. TF-IDF y similitud libro-libro
El vectorizador limita vocabulario y usa bigramas. La matriz se mantiene dispersa; solo se densifica un vector de similitudes de un libro contra el catálogo para mostrar ejemplos.

In [3]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=80_000, sublinear_tf=True, dtype=np.float32)
matrix = vectorizer.fit_transform(content.content_text)
content['content_row'] = np.arange(len(content), dtype=np.int64)
print(f'Matriz TF-IDF: {matrix.shape}; densidad={matrix.nnz / (matrix.shape[0] * matrix.shape[1]):.6%}')
joblib.dump(vectorizer, ARTIFACTS / 'tfidf_vectorizer.joblib'); sp.save_npz(ARTIFACTS / 'book_tfidf.npz', matrix); content[['book_id', 'content_row']].to_parquet(ARTIFACTS / 'content_book_mapping.parquet', index=False)
(ARTIFACTS / 'config.json').write_text(json.dumps({'ngram_range': [1, 2], 'min_df': 2, 'max_features': 80000, 'positive_rating': 4, 'fields': ['title', 'description', 'authors', 'thematic_genres', 'source_genre', 'language_code']}, indent=2), encoding='utf-8')
reference_row = int(np.flatnonzero(content.description.ne('').to_numpy())[0]); similarities = (matrix[reference_row] @ matrix.T).toarray().ravel(); similar_rows = np.argsort(-similarities)[1:11]
display(content.loc[similar_rows, ['book_id', 'title', 'authors_text', 'genres_text']].assign(similitud=similarities[similar_rows]))


Matriz TF-IDF: (125871, 80000); densidad=0.125127%


,book_id,title,authors_text,genres_text,similitud
60969,25119864,تفاصيل أخرى للماء,slTn lsbhn,poetry,0.472123
3440,10800181,النهاية والبداية وقصائد أخرى,Wislawa Szymborska fyswf shymbwrsk htf ljnby,"comics, graphic history, historical fiction, b...",0.432134
23922,16047601,باریس گادونوف,Alexander Pushkin abtyn glkhr,"fiction history, historical fiction, biography...",0.425197
20211,15354299,بوريس جودونوف,'lksndr bwshkyn Alexander Pushkin,"fiction history, historical fiction, biography...",0.416751
25592,16166024,التراجيديات الصغيرة,'lksndr bwshkyn Alexander Pushkin drys lmlyny,"fiction history, historical fiction, biography...",0.394731
14631,13488614,النهاية و البداية وقصائد أخرى,Wislawa Szymborska fyswf shymbwrsk,"comics, graphic history, historical fiction, b...",0.391854
117535,8076394,Каменный гость,Alexander Pushkin,"fantasy, paranormal fiction history, historica...",0.376375
54577,23358215,الروح جدًا مرهقة,slm Hjy,fiction poetry romance,0.367034
22405,15831508,أبو نواس فى نوادره و بعض قصائده,'bw nws slm shms ldyn,"history, historical fiction, biography poetry",0.340878
95046,35967656,Евгениј Онегин,Alexander Pushkin,"fiction history, historical fiction, biography...",0.335433


## 3. Perfiles desde entrenamiento y candidatos comparables
Los perfiles usan únicamente ratings de entrenamiento de al menos cuatro estrellas. Para mantener comparabilidad, se persisten exactamente los mismos candidatos para popularidad, factorización y contenido: un relevante retenido más 1,000 ítems no vistos por usuario.

In [4]:
train = pd.read_parquet(ML / 'train.parquet'); valid = pd.read_parquet(ML / 'valid.parquet'); test = pd.read_parquet(ML / 'test.parquet')
book_mapping = pd.read_parquet(ML / 'book_mapping.parquet'); user_mapping = pd.read_parquet(ML / 'user_mapping.parquet')
idx_to_row = book_mapping.merge(content[['book_id', 'content_row']], on='book_id', how='left').sort_values('book_idx').content_row.to_numpy(dtype=np.int64)
assert len(idx_to_row) == len(book_mapping) and not pd.isna(idx_to_row).any()
valid_candidates = make_candidates(train, valid, len(book_mapping), SEED + 10, MAX_USERS, NEGATIVES, RELEVANCE)
test_candidates = make_candidates(train, test, len(book_mapping), SEED + 11, MAX_USERS, NEGATIVES, RELEVANCE)
valid_candidates.to_parquet(ML / 'evaluation_v2_valid.parquet', index=False); test_candidates.to_parquet(ML / 'evaluation_v2_test.parquet', index=False)
print(f'Candidatos validación/prueba: {len(valid_candidates):,}/{len(test_candidates):,}')


Candidatos validación/prueba: 5,005,000/5,005,000


## 4. Comparación consistente: contenido, colaborativo y popularidad
El checkpoint colaborativo de referencia no se reentrena. Las tres puntuaciones se calculan sobre las mismas filas de candidatos, usuarios, exclusiones y relevancia. Se reportan además segmentos por actividad del usuario y popularidad del ítem relevante.

In [5]:
collab_dir = ROOT / 'artifacts' / 'collaborative_mf'; config = json.loads((collab_dir / 'config.json').read_text())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MatrixFactorization(config['num_users'], config['num_items'], config['factors'], config['global_mean']).to(device)
model.load_state_dict(torch.load(collab_dir / 'best_model.pt', map_location=device, weights_only=True)['state_dict']); model.eval()
popularity = train.groupby('book_idx').size().reindex(range(len(book_mapping)), fill_value=0).to_numpy(dtype=np.float32)
def collaborative_scores(users, items):
    values = np.empty(len(users), dtype=np.float32)
    for start in range(0, len(users), 262_144):
        stop = start + 262_144
        with torch.no_grad(): values[start:stop] = model(torch.tensor(users[start:stop], device=device), torch.tensor(items[start:stop], device=device)).cpu().numpy()
    return values
def score_all(frame):
    result = score_candidates(frame, collaborative_scores, 'collaborative_score')
    result['popularity_score'] = popularity[result.book_idx.to_numpy()]
    result['content_score'] = content_candidate_scores(result, train, idx_to_row, matrix, positive_rating=RELEVANCE)
    return result
valid_scores, test_scores = score_all(valid_candidates), score_all(test_candidates)
for split, frame in [('valid', valid_scores), ('test', test_scores)]: frame.to_parquet(ARTIFACTS / f'{split}_candidate_scores.parquet', index=False)
rows = []; per_user = {}
for partition, frame in [('validación', valid_scores), ('prueba', test_scores)]:
    for label, column in [('Popularidad', 'popularity_score'), ('Colaborativo', 'collaborative_score'), ('Contenido', 'content_score')]:
        detail, metrics = ranking_metrics(frame, column, TOP_K); detail['modelo'] = label; detail['partición'] = partition; per_user[(partition, label)] = detail
        rows.append({'partición': partition, 'modelo': label, 'usuarios': len(detail), **metrics})
comparison = pd.DataFrame(rows); display(comparison)
segment_rows = []
for label in ['Popularidad', 'Colaborativo', 'Contenido']:
    detail = per_user[('validación', label)].copy(); detail['actividad'] = pd.qcut(detail.user_train_interactions, 3, duplicates='drop'); detail['popularidad_objetivo'] = pd.qcut(detail.target_train_popularity, 3, duplicates='drop')
    for dimension in ['actividad', 'popularidad_objetivo']:
        grouped = detail.groupby(dimension, observed=True)[['recall_at_k', 'ndcg_at_k']].mean().reset_index(); grouped['modelo'] = label; grouped['segmento'] = dimension; segment_rows.append(grouped)
display(pd.concat(segment_rows, ignore_index=True))
broad_valid = score_all(make_candidates(train, valid, len(book_mapping), SEED + 12, max_users=1000, negatives=5000, relevance_rating=RELEVANCE))
broad_rows = []
for label, column in [('Popularidad', 'popularity_score'), ('Colaborativo', 'collaborative_score'), ('Contenido', 'content_score')]:
    _, metrics = ranking_metrics(broad_valid, column, TOP_K); broad_rows.append({'protocolo': 'validación ampliada: 1 relevante + 5000 no vistos', 'modelo': label, 'usuarios': broad_valid.user_idx.nunique(), **metrics})
broad_results = pd.DataFrame(broad_rows); display(broad_results); broad_valid.to_parquet(ARTIFACTS / 'valid_broad_candidate_scores.parquet', index=False)
comparison.to_json(ARTIFACTS / 'evaluation_v2.json', orient='records', indent=2, force_ascii=False)
display(Markdown('**Interpretación.** Los segmentos muestran dónde el contenido responde mejor o peor según historial y popularidad. No se ajustan parámetros con prueba; sus métricas se reservan para comparar los modelos ya fijados bajo candidatos compartidos.'))


,partición,modelo,usuarios,precision_at_k,recall_at_k,ndcg_at_k
0,validación,Popularidad,5000,0.04034,0.4034,0.265721
1,validación,Colaborativo,5000,0.01450,0.1450,0.087826
2,validación,Contenido,5000,0.04106,0.4106,0.285293
3,prueba,Popularidad,5000,0.03732,0.3732,0.245294
4,prueba,Colaborativo,5000,0.01372,0.1372,0.079582
5,prueba,Contenido,5000,0.03142,0.3142,0.209847


,actividad,recall_at_k,ndcg_at_k,modelo,segmento,popularidad_objetivo
0,"(2.999, 6.0]",0.483607,0.337569,Popularidad,actividad,NaN
1,"(6.0, 18.0]",0.421586,0.270608,Popularidad,actividad,NaN
2,"(18.0, 3861.0]",0.299690,0.184661,Popularidad,actividad,NaN
3,NaN,0.000000,0.000000,Popularidad,popularidad_objetivo,"(0.999, 127.0]"
4,NaN,0.212012,0.071683,Popularidad,popularidad_objetivo,"(127.0, 1288.0]"
5,NaN,0.998800,0.725843,Popularidad,popularidad_objetivo,"(1288.0, 27295.0]"
6,"(2.999, 6.0]",0.136417,0.083732,Colaborativo,actividad,NaN
7,"(6.0, 18.0]",0.150268,0.091937,Colaborativo,actividad,NaN
8,"(18.0, 3861.0]",0.148607,0.087888,Colaborativo,actividad,NaN
9,NaN,0.024566,0.011137,Colaborativo,popularidad_objetivo,"(0.999, 127.0]"


,protocolo,modelo,usuarios,precision_at_k,recall_at_k,ndcg_at_k
0,validación ampliada: 1 relevante + 5000 no vistos,Popularidad,1000,0.0209,0.209,0.123957
1,validación ampliada: 1 relevante + 5000 no vistos,Colaborativo,1000,0.0059,0.059,0.036622
2,validación ampliada: 1 relevante + 5000 no vistos,Contenido,1000,0.0232,0.232,0.149828


**Interpretación.** Los segmentos muestran dónde el contenido responde mejor o peor según historial y popularidad. No se ajustan parámetros con prueba; sus métricas se reservan para comparar los modelos ya fijados bajo candidatos compartidos.

## 5. Inferencia
Para un libro se calcula similitud contra filas TF-IDF; para un usuario conocido se usa su perfil positivo de entrenamiento. Un usuario nuevo requiere preferencias explícitas, que se tratarán en el híbrido.